# 03 · Business Insights
**Saudi Tourism & Hospitality Establishments Analytics – Q2 2026**

Turns the EDA into decision-oriented insights. Each insight has four parts: observation, supporting number, likely explanation (a hypothesis, the data does not prove causes) and recommendation. The same text is exported to `reports/business_insights.md`. The notebook also produces the Python reference values used to validate the Power BI measures and cross-checks them with SQL.

In [1]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from IPython.display import Markdown, display
ROOT = Path.cwd().parent; D = ROOT/'data'/'processed'; CH = ROOT/'reports'/'charts'; CH.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, INK, INK2, MUTED, GRID, SURF = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#fcfcfb'
HOT, APT, ALL = 'Hotels', 'Serviced apartments & other', 'All facilities'
sns.set_theme(style='white')
plt.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF, 'figure.dpi': 110, 'savefig.dpi': 160,
    'font.family': 'DejaVu Sans', 'font.size': 10, 'text.color': INK, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': INK2,
    'axes.edgecolor': '#c3c2b7', 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False,
    'axes.titlesize': 12.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlepad': 14})
def finish(ax, name, grid='y', sub=None):
    if grid: ax.grid(axis=grid, color=GRID, lw=0.8); ax.set_axisbelow(True)
    if sub: ax.set_title(ax.get_title(), pad=24); ax.text(0, 1.02, sub, transform=ax.transAxes, color=INK2, fontsize=9.5)
    ax.figure.tight_layout(); ax.figure.savefig(CH/name, bbox_inches='tight'); plt.show()
def note(t): display(Markdown('> **What it means for the business:** ' + t))
pct = lambda v, d=1: f'{v*100:.{d}f}%'
q = pd.read_csv(D/'Fact_Quarterly.csv').merge(pd.read_csv(D/'Dim_Quarter.csv'), on='Quarter_Key')
reg = pd.read_csv(D/'Fact_Regional_Hospitality.csv'); regm = pd.read_csv(D/'Fact_Regional_Monthly.csv')
city = pd.read_csv(D/'Fact_City_Hospitality.csv')
ea = pd.read_csv(D/'Fact_Employment_Activity.csv').merge(pd.read_csv(D/'Dim_Activity.csv'), on='Activity')
er = pd.read_csv(D/'Fact_Employment_Region.csv')
sa = pd.read_csv(D/'Fact_Establishments_Activity.csv').merge(pd.read_csv(D/'Dim_Activity.csv'), on='Activity')
sr = pd.read_csv(D/'Fact_Establishments_Region.csv')
rh, ra = reg[reg.Facility_Type == HOT].set_index('Region'), reg[reg.Facility_Type == APT].set_index('Region')
last, yago = q.iloc[-1], q.iloc[-5]


In [2]:
K = {}
wavg = lambda v, w: float((v*w).sum()/w.sum())
for nm, g in ((HOT, rh), (APT, ra)):
    K[nm] = dict(occ=wavg(g.Occupancy, g.Available_Rooms), adr=wavg(g.ADR, g.Occupancy*g.Available_Rooms),
                 alos_simple=float(g.ALOS.mean()), occ_simple=float(g.Occupancy.mean()), adr_simple=float(g.ADR.mean()))
reg['RevPAR'] = reg.Occupancy*reg.ADR; reg['RevPAR_2025'] = reg.Occupancy_Q2_2025*reg.ADR_Q2_2025
rv = reg[reg.Facility_Type == HOT].set_index('Region'); rv['RevPAR_chg'] = rv.RevPAR/rv.RevPAR_2025-1
occ_d = (rh.Occupancy-rh.Occupancy_Q2_2025)*100
d = q.dropna(subset=['Occupancy_Hotels']); q2 = d[d.Quarter_Num == 2].ADR_Hotels; oth = d[d.Quarter_Num != 2].ADR_Hotels
est_a = sa.set_index('Activity_Short').Establishments; emp_a = ea.groupby('Activity_Short').Employees.sum()
tot_e = int(ea.Employees.sum()); tot_s = int(est_a.sum())
sau = ea[ea.Nationality == 'Saudi']; s_by_act = sau.groupby('Activity_Short').Employees.sum()/emp_a
er_p = er.pivot_table(index='Region', columns='Nationality', values='Employees', aggfunc='sum'); er_p['Saudi_share'] = er_p['Saudi']/er_p.sum(axis=1)
mk = regm[(regm.Region == 'Makkah Al Mukarramah') & (regm.Facility_Type == HOT)].set_index('Month')
non_holy = rh.ALOS.drop(['Makkah Al Mukarramah', 'Al Madinah Al Munawwarah'])
fac0 = q.Licensed_Facilities.dropna()
rv[['Occupancy', 'ADR', 'RevPAR', 'RevPAR_2025', 'RevPAR_chg']].sort_values('RevPAR', ascending=False).round(3)

,Occupancy,ADR,RevPAR,RevPAR_2025,RevPAR_chg
Region,,,,,
Al Madinah Al Munawwarah,0.653,589.459,385.122,425.471,-0.095
Makkah Al Mukarramah,0.500,585.311,292.364,345.638,-0.154
Ar Riyadh,0.427,540.912,230.824,385.066,-0.401
Tabuk,0.286,738.422,211.283,285.334,-0.260
Eastern region,0.478,433.089,206.977,246.663,-0.161
Al Qaseem,0.427,389.625,166.350,181.161,-0.082
Al Bahah,0.343,331.109,113.559,103.261,0.100
Aseer,0.380,297.060,112.874,124.617,-0.094
Jazan,0.364,290.264,105.741,108.503,-0.025


RevPAR here is a proxy: `occupancy x ADR` (revenue per available room-night, SAR). The table above is for hotels.

In [3]:
I = []
def add(title, obs, num, why, rec): I.append(dict(title=title, obs=obs, num=num, why=why, rec=rec))
add('Food & beverage is the backbone of the tourism economy',
 'One activity dominates both the number of establishments and the jobs.',
 f'Food & beverage = {est_a["Food & beverage"]:,.0f} establishments ({pct(est_a["Food & beverage"]/tot_s)}) and {emp_a["Food & beverage"]:,.0f} employees ({pct(emp_a["Food & beverage"]/tot_e)}), but only {emp_a["Food & beverage"]/est_a["Food & beverage"]:.1f} employees per establishment. Cultural activities are {pct(est_a["Cultural"]/tot_s)} of establishments.',
 'Restaurants and cafes have low entry barriers and serve residents as well as visitors, so the count is inflated relative to pure tourism demand.',
 'Treat F&B as a small-business segment (licensing simplification, training, quality programmes). Experience-based activities (cultural, sports & recreation) are a very small base and the clearest diversification gap.')
add('Supply is growing faster than room demand',
 'Licensed facilities keep rising while occupancy falls for both facility types.',
 f'Licensed hospitality facilities: {yago.Licensed_Facilities:,.0f} (Q2 2025) to {last.Licensed_Facilities:,.0f} (Q2 2026), +{pct(last.Licensed_Facilities/yago.Licensed_Facilities-1)}. Hotel occupancy {pct(yago.Occupancy_Hotels)} to {pct(last.Occupancy_Hotels)}; serviced apartments {pct(yago.Occupancy_Apartments)} to {pct(last.Occupancy_Apartments)}, the lowest in 14 quarters.',
 'New licensed capacity (partly formalisation of existing properties) is being added faster than guest nights. Part of the facility growth may be licensing of already-operating units rather than new rooms.',
 'Shift emphasis from adding capacity to filling it: demand campaigns for low-occupancy regions and for serviced apartments. Track occupancy together with new licences before approving further expansion incentives.')
add('Hotel occupancy differs by more than 2x between regions',
 'A few regions are busy; most run well below the national figure.',
 f'{rh.Occupancy.idxmax()} {pct(rh.Occupancy.max())} vs {rh.Occupancy.idxmin()} {pct(rh.Occupancy.min())}. {int((occ_d < 0).sum())} of 13 regions declined year-on-year. Makkah + Madinah hold {pct(rh.loc[["Makkah Al Mukarramah","Al Madinah Al Munawwarah"]].Available_Rooms.sum()/rh.Available_Rooms.sum())} of hotel rooms, so the national rate ({pct(K[HOT]["occ"])}) is above the rate of {int((rh.Occupancy < K[HOT]["occ"]).sum())} regions.',
 'Religious tourism gives the two holy-city regions a stable demand base that other regions do not have; smaller regions depend on domestic and seasonal travel.',
 'Never manage by the national average. Set occupancy targets per region and prioritise the regions below 35% (Tabuk, Hail, Al Jawf, Al Bahah) for seasonal events and domestic packages.')
add('Riyadh hotels had a sharp price and volume correction',
 'The capital lost both rate and occupancy in the same quarter.',
 f'Riyadh hotel ADR SAR {rh.ADR_Q2_2025["Ar Riyadh"]:,.0f} to {rh.ADR["Ar Riyadh"]:,.0f} ({pct(rh.ADR["Ar Riyadh"]/rh.ADR_Q2_2025["Ar Riyadh"]-1)}); occupancy {pct(rh.Occupancy_Q2_2025["Ar Riyadh"])} to {pct(rh.Occupancy["Ar Riyadh"])} ({occ_d["Ar Riyadh"]:+.1f} pts). RevPAR proxy SAR {rv.RevPAR_2025["Ar Riyadh"]:,.0f} to {rv.RevPAR["Ar Riyadh"]:,.0f} ({pct(rv.RevPAR_chg["Ar Riyadh"])}).',
 'Possible causes: new hotel openings, fewer large events or conferences than in Q2 2025, or a change in the mix of licensed hotels toward cheaper categories. The data cannot separate these.',
 'Investigate Riyadh first: compare event calendars and new room supply for the two quarters. Operators should review pricing strategy; a rate cut that does not lift occupancy points to an oversupply problem, not a price problem.')
add('Hotel pricing depends on the Hajj quarter, and the premium is shrinking',
 'ADR jumps every Q2 and falls back, but each Q2 peak is lower than the last.',
 f'Average hotel ADR in Q2 = SAR {q2.mean():,.0f} vs SAR {oth.mean():,.0f} in other quarters (+{pct(q2.mean()/oth.mean()-1, 0)}). Q2 peak: SAR {q2.iloc[0]:,.0f} (2023), {q2.iloc[1]:,.0f}, {q2.iloc[2]:,.0f}, {q2.iloc[3]:,.0f} (2026). Makkah hotels in May 2026: ADR SAR {mk.ADR["May"]:,.0f}, stay {mk.ALOS["May"]:.1f} nights, vs SAR {mk.ADR["June"]:,.0f} in June.',
 'Hajj fell in Q2 in all four years. The declining peak is consistent with more room supply in Makkah and Madinah and with growth of lower-priced licensed hotels diluting the average.',
 'Do not use Q2 as a baseline for annual planning. Build shoulder-season demand (Umrah outside peak, combined Makkah-Madinah-Jeddah itineraries) to reduce reliance on one month.')
add('Price levels vary 3.7x across regions; Tabuk is high-price / low-volume',
 'The most expensive hotel market is also the emptiest.',
 f'Hotel ADR: {rh.ADR.idxmax()} SAR {rh.ADR.max():,.0f} with {pct(rh.Occupancy["Tabuk"])} occupancy ({occ_d["Tabuk"]:+.1f} pts year-on-year); {rh.ADR.idxmin()} SAR {rh.ADR.min():,.0f}. Hotels charge {K[HOT]["adr"]/K[APT]["adr"]:.1f}x serviced apartments (SAR {K[HOT]["adr"]:,.0f} vs {K[APT]["adr"]:,.0f}).',
 'Tabuk region includes new luxury resort destinations on the Red Sea coast, which raise the average rate while volumes are still small. This is an assumption based on the region, not something in the data.',
 'Add mid-priced supply and packages in Tabuk to widen the customer base; in low-ADR regions (Najran, Northern Borders, Al Jawf) focus on quality upgrades before price increases.')
add('Outside the holy cities, guests stay less than two nights',
 'Length of stay is long only where the trip is a pilgrimage.',
 f'Makkah hotels {rh.ALOS["Makkah Al Mukarramah"]:.1f} nights, Madinah {rh.ALOS["Al Madinah Al Munawwarah"]:.1f}; median of the other 11 regions {non_holy.median():.1f} nights (range {non_holy.min():.1f}-{non_holy.max():.1f}). Serviced apartments: {last.ALOS_Apartments:.1f} nights nationally.',
 'Most non-religious trips are short business, family-visit or transit stays.',
 'Extending the stay by even half a night is the cheapest revenue lever in these regions: bundle accommodation with attractions and events, and promote multi-city routes.')
add('The workforce is growing, mostly through non-Saudi hiring',
 'Saudi participation is low on average and very uneven across activities and regions.',
 f'{tot_e:,.0f} employees (+{pct(last.Total_Employees/yago.Total_Employees-1)} year-on-year): non-Saudi +{pct(last.NonSaudi_Employees/yago.NonSaudi_Employees-1)}, Saudi +{pct(last.Saudi_Employees/yago.Saudi_Employees-1)}. Saudi share {pct(sau.Employees.sum()/tot_e)}: {pct(s_by_act.min(), 0)} in {s_by_act.idxmin()} vs {pct(s_by_act.max(), 0)} in {s_by_act.idxmax()}; by region {pct(er_p.Saudi_share.min())} ({er_p.Saudi_share.idxmin()}) to {pct(er_p.Saudi_share.max())} ({er_p.Saudi_share.idxmax()}). Women are {pct(ea[ea.Gender=="Female"].Employees.sum()/tot_e)} of all employees but {pct(sau[sau.Gender=="Female"].Employees.sum()/sau.Employees.sum())} of Saudi employees.',
 'Labour-intensive, lower-wage activities (F&B, road transport) rely on expatriate workers; regulated transport activities (rail, air) have high Saudi shares.',
 'Target localisation programmes at F&B (largest employer, lowest Saudi share) and at the regions below 13%. Saudi women are already nearly half of Saudi tourism staff, so female-focused training and scheduling policies have a large reach.')
md = ['# Business Insights – Saudi Tourism & Hospitality Establishments, Q2 2026', '',
      '_Generated by `notebooks/03_business_insights.ipynb`. All numbers are computed from `data/processed/`. Explanations are hypotheses; the data shows what changed, not why._', '']
for n, i in enumerate(I, 1):
    md += [f'## {n}. {i["title"]}', '', f'1. **Observation:** {i["obs"]}', f'2. **Supporting numbers:** {i["num"]}', f'3. **Likely explanation:** {i["why"]}', f'4. **Recommendation:** {i["rec"]}', '']
(ROOT/'reports'/'business_insights.md').write_text('\n'.join(md), encoding='utf-8')
display(Markdown('\n'.join(md)))

# Business Insights – Saudi Tourism & Hospitality Establishments, Q2 2026

_Generated by `notebooks/03_business_insights.ipynb`. All numbers are computed from `data/processed/`. Explanations are hypotheses; the data shows what changed, not why._

## 1. Food & beverage is the backbone of the tourism economy

1. **Observation:** One activity dominates both the number of establishments and the jobs.
2. **Supporting numbers:** Food & beverage = 115,697 establishments (64.5%) and 654,758 employees (61.2%), but only 5.7 employees per establishment. Cultural activities are 0.2% of establishments.
3. **Likely explanation:** Restaurants and cafes have low entry barriers and serve residents as well as visitors, so the count is inflated relative to pure tourism demand.
4. **Recommendation:** Treat F&B as a small-business segment (licensing simplification, training, quality programmes). Experience-based activities (cultural, sports & recreation) are a very small base and the clearest diversification gap.

## 2. Supply is growing faster than room demand

1. **Observation:** Licensed facilities keep rising while occupancy falls for both facility types.
2. **Supporting numbers:** Licensed hospitality facilities: 5,326 (Q2 2025) to 6,278 (Q2 2026), +17.9%. Hotel occupancy 53.2% to 51.0%; serviced apartments 50.2% to 46.5%, the lowest in 14 quarters.
3. **Likely explanation:** New licensed capacity (partly formalisation of existing properties) is being added faster than guest nights. Part of the facility growth may be licensing of already-operating units rather than new rooms.
4. **Recommendation:** Shift emphasis from adding capacity to filling it: demand campaigns for low-occupancy regions and for serviced apartments. Track occupancy together with new licences before approving further expansion incentives.

## 3. Hotel occupancy differs by more than 2x between regions

1. **Observation:** A few regions are busy; most run well below the national figure.
2. **Supporting numbers:** Al Madinah Al Munawwarah 65.3% vs Tabuk 28.6%. 9 of 13 regions declined year-on-year. Makkah + Madinah hold 86.3% of hotel rooms, so the national rate (51.0%) is above the rate of 12 regions.
3. **Likely explanation:** Religious tourism gives the two holy-city regions a stable demand base that other regions do not have; smaller regions depend on domestic and seasonal travel.
4. **Recommendation:** Never manage by the national average. Set occupancy targets per region and prioritise the regions below 35% (Tabuk, Hail, Al Jawf, Al Bahah) for seasonal events and domestic packages.

## 4. Riyadh hotels had a sharp price and volume correction

1. **Observation:** The capital lost both rate and occupancy in the same quarter.
2. **Supporting numbers:** Riyadh hotel ADR SAR 739 to 541 (-26.8%); occupancy 52.1% to 42.7% (-9.4 pts). RevPAR proxy SAR 385 to 231 (-40.1%).
3. **Likely explanation:** Possible causes: new hotel openings, fewer large events or conferences than in Q2 2025, or a change in the mix of licensed hotels toward cheaper categories. The data cannot separate these.
4. **Recommendation:** Investigate Riyadh first: compare event calendars and new room supply for the two quarters. Operators should review pricing strategy; a rate cut that does not lift occupancy points to an oversupply problem, not a price problem.

## 5. Hotel pricing depends on the Hajj quarter, and the premium is shrinking

1. **Observation:** ADR jumps every Q2 and falls back, but each Q2 peak is lower than the last.
2. **Supporting numbers:** Average hotel ADR in Q2 = SAR 658 vs SAR 432 in other quarters (+52%). Q2 peak: SAR 756 (2023), 669, 643, 563 (2026). Makkah hotels in May 2026: ADR SAR 963, stay 11.9 nights, vs SAR 291 in June.
3. **Likely explanation:** Hajj fell in Q2 in all four years. The declining peak is consistent with more room supply in Makkah and Madinah and with growth of lower-priced licensed hotels diluting the average.
4. **Recommendation:** Do not use Q2 as a baseline for annual planning. Build shoulder-season demand (Umrah outside peak, combined Makkah-Madinah-Jeddah itineraries) to reduce reliance on one month.

## 6. Price levels vary 3.7x across regions; Tabuk is high-price / low-volume

1. **Observation:** The most expensive hotel market is also the emptiest.
2. **Supporting numbers:** Hotel ADR: Tabuk SAR 738 with 28.6% occupancy (-11.1 pts year-on-year); Najran SAR 200. Hotels charge 2.9x serviced apartments (SAR 573 vs 198).
3. **Likely explanation:** Tabuk region includes new luxury resort destinations on the Red Sea coast, which raise the average rate while volumes are still small. This is an assumption based on the region, not something in the data.
4. **Recommendation:** Add mid-priced supply and packages in Tabuk to widen the customer base; in low-ADR regions (Najran, Northern Borders, Al Jawf) focus on quality upgrades before price increases.

## 7. Outside the holy cities, guests stay less than two nights

1. **Observation:** Length of stay is long only where the trip is a pilgrimage.
2. **Supporting numbers:** Makkah hotels 7.3 nights, Madinah 4.3; median of the other 11 regions 1.8 nights (range 1.6-2.5). Serviced apartments: 2.1 nights nationally.
3. **Likely explanation:** Most non-religious trips are short business, family-visit or transit stays.
4. **Recommendation:** Extending the stay by even half a night is the cheapest revenue lever in these regions: bundle accommodation with attractions and events, and promote multi-city routes.

## 8. The workforce is growing, mostly through non-Saudi hiring

1. **Observation:** Saudi participation is low on average and very uneven across activities and regions.
2. **Supporting numbers:** 1,069,120 employees (+7.2% year-on-year): non-Saudi +8.2%, Saudi +4.2%. Saudi share 24.1%: 18% in Food & beverage vs 68% in Rail transport; by region 9.6% (Najran) to 28.9% (Ar Riyadh). Women are 13.2% of all employees but 46.6% of Saudi employees.
3. **Likely explanation:** Labour-intensive, lower-wage activities (F&B, road transport) rely on expatriate workers; regulated transport activities (rail, air) have high Saudi shares.
4. **Recommendation:** Target localisation programmes at F&B (largest employer, lowest Saudi share) and at the regions below 13%. Saudi women are already nearly half of Saudi tourism staff, so female-focused training and scheduling policies have a large reach.


## Reference values for validating the Power BI measures
National occupancy and ADR are **weighted** averages (occupancy by available rooms; ADR by rooms sold = rooms x occupancy). The weighted results reproduce the officially published national figures closely, which a simple average of 13 regions does not.

In [4]:
fem = int(ea[ea.Gender == 'Female'].Employees.sum())
ref = pd.DataFrame([
 ('Total Establishments', tot_s, int(last.Tourism_Establishments)),
 ('Total Employees', tot_e, int(last.Total_Employees)),
 ('Male Employees', tot_e-fem, np.nan), ('Female Employees', fem, np.nan),
 ('Female Employee Share %', fem/tot_e, last.Female_Share),
 ('Saudi Employees', int(sau.Employees.sum()), int(last.Saudi_Employees)),
 ('Licensed Facilities', int(reg.Facilities.sum()), int(last.Licensed_Facilities)),
 ('Available Rooms', int(reg.Available_Rooms.sum()), np.nan),
 ('Average Hotel Occupancy Rate (room-weighted)', K[HOT]['occ'], last.Occupancy_Hotels),
 ('Average Apartment Occupancy Rate (room-weighted)', K[APT]['occ'], last.Occupancy_Apartments),
 ('Average Hotel ADR (weighted by rooms sold)', K[HOT]['adr'], last.ADR_Hotels),
 ('Average Apartment ADR (weighted by rooms sold)', K[APT]['adr'], last.ADR_Apartments),
 ('Hotel Occupancy - simple average of regions', K[HOT]['occ_simple'], last.Occupancy_Hotels),
 ('Hotel ADR - simple average of regions', K[HOT]['adr_simple'], last.ADR_Hotels),
 ('Average Hotel Length of Stay - national (quarterly table)', last.ALOS_Hotels, last.ALOS_Hotels),
], columns=['Measure', 'Python (from regional/activity tables)', 'Published national value (Quarterly_Trends, Q2 2026)'])
ref['Difference %'] = (ref.iloc[:, 1]/ref.iloc[:, 2]-1)*100
ref.to_csv(ROOT/'reports'/'measure_validation_python.csv', index=False); ref.round(4)

,Measure,Python (from regional/activity tables),"Published national value (Quarterly_Trends, Q2 2026)",Difference %
0,Total Establishments,1.795070e+05,1.795070e+05,0.0000
1,Total Employees,1.069120e+06,1.069120e+06,0.0000
2,Male Employees,9.284070e+05,NaN,NaN
3,Female Employees,1.407130e+05,NaN,NaN
4,Female Employee Share %,1.316000e-01,1.320000e-01,-0.2663
5,Saudi Employees,2.580430e+05,2.580430e+05,0.0000
6,Licensed Facilities,6.278000e+03,6.278000e+03,0.0000
7,Available Rooms,6.228980e+05,NaN,NaN
8,Average Hotel Occupancy Rate (room-weighted),5.101000e-01,5.099000e-01,0.0425
9,Average Apartment Occupancy Rate (room-weighted),4.601000e-01,4.650000e-01,-1.0456


Totals match exactly. Room-weighted hotel occupancy differs from the published value by a few hundredths of a point and weighted ADR by under 2%, while the simple averages are off by roughly 20–30%. **This is why the Power BI measures use weighted averages.** Length of stay cannot be weighted correctly (it needs guest arrivals, which are not in the data), so the national value is taken from the quarterly table.

## SQL cross-check
The same figures recomputed with SQL (SQLite in memory) from the processed CSVs, using the queries in `sql/analysis_queries.sql`.

In [5]:
import sqlite3
con = sqlite3.connect(':memory:')
for f in sorted(D.glob('*.csv')):
    if f.stem.startswith(('Fact_', 'Dim_')): pd.read_csv(f).to_sql(f.stem, con, index=False)
sql = (ROOT/'sql'/'analysis_queries.sql').read_text()
for block in [b for b in sql.split(';') if b.strip()]:
    title = [l for l in block.strip().splitlines() if l.startswith('--')][0][3:]
    print('\n' + title); display(pd.read_sql(block, con).round(4))


Q1. Headline totals (each fact table at its own grain, no joins between facts)


,total_establishments,total_employees,male_employees,female_employees,total_employees_region_check
0,179507,1069120,928407,140713,1069120



Q2. National occupancy and ADR by facility type (weighted)


,Facility_Type,facilities,rooms,occupancy_weighted,adr_weighted,occupancy_simple_avg
0,Hotels,3020,509394,0.5101,573.0854,0.4117
1,Serviced apartments & other,3258,113504,0.4601,197.7839,0.4275



Q3. Hotel performance by region with year-on-year change and RevPAR proxy


,Region,Available_Rooms,occupancy_pct,occupancy_change_pts,adr_sar,adr_change_pct,revpar_sar,length_of_stay
0,Al Madinah Al Munawwarah,73798,65.3,-1.5,589.0,-7.4,385.0,4.30
1,Makkah Al Mukarramah,365584,50.0,-0.8,585.0,-14.0,292.0,7.29
2,Eastern region,17317,47.8,-7.0,433.0,-3.8,207.0,2.10
3,Northern Borders,1058,46.8,1.2,207.0,13.7,97.0,2.29
4,Al Qaseem,2028,42.7,0.7,390.0,-9.7,166.0,1.67
5,Ar Riyadh,29134,42.7,-9.4,541.0,-26.8,231.0,2.13
6,Najran,1662,39.9,-2.4,200.0,-15.2,80.0,1.61
7,Aseer,5134,38.0,-1.3,297.0,-6.2,113.0,1.83
8,Jazan,4425,36.4,1.4,290.0,-6.3,106.0,1.75
9,Al Bahah,644,34.3,5.2,331.0,-6.6,114.0,1.85


... (4 outputs left out)


Q6. City ranking by occupancy (all facilities)


,City,occupancy_pct,adr_sar,length_of_stay
0,Al-Madinah Al-Munawwarah city,64.4,575.0,4.28
1,Jiddah governorate,54.4,365.0,2.21
2,Najran city,51.4,144.0,2.13
3,Dammam metropolitan area,50.8,342.0,2.09
4,Makkah Al-Mukarramah city,49.1,598.0,8.89
5,Riyadh city,48.3,362.0,2.13
6,Bariduh city and Unayzah governorate,48.2,227.0,1.69
7,Arar city,46.3,209.0,1.54
8,Abha city and Khamis Mushayt governorate,43.1,200.0,1.84
9,Jazan city,40.1,271.0,1.79



Q7. Quarterly trend with year-on-year growth of establishments


,Quarter_Label,Tourism_Establishments,establishments_yoy_pct,hotel_occupancy_pct,hotel_adr_sar,Total_Employees
0,2022 Q1,109681,NaN,NaN,NaN,NaN
1,2022 Q2,115366,NaN,NaN,NaN,NaN
2,2022 Q3,118087,NaN,NaN,NaN,NaN
3,2022 Q4,121779,NaN,NaN,NaN,NaN
4,2023 Q1,124977,13.9,63.2,515.0,900532.0
5,2023 Q2,127767,10.7,54.9,756.0,912197.0
6,2023 Q3,130379,10.4,53.4,443.0,919172.0
7,2023 Q4,134098,10.1,60.2,447.0,929404.0
8,2024 Q1,138267,10.6,60.9,494.0,944299.0
9,2024 Q2,141585,10.8,55.4,669.0,959175.0
